# Trust-Align Supervised Fine-Tuning (SFT)
### Model: Qwen3-4B-Instruct-2507  
### Dataset: declare-lab/Trust-Data

這份 Notebook 是可信任 RAG alignment pipeline 的第一階段(SFT)。  
全部的 alignment pipeline 如下：

```text
Qwen3-4B-Instruct-2507
         │
         ▼
trust_align_sft（學習 grounded citation 與適當拒答）
         │
         ▼
SFT LoRA adapter
         │
         ▼
trust_align_dpo（偏好對齊）
```

本 Notebook 架構：

1. 固定模型與資料集 revision。
2. 載入 Trust-Data 官方 `trust_align_sft` train/test split。
3. 使用 Qwen3 chat template，且只對 assistant tokens 計算 SFT loss。
4. 在 tokenization 前排除超長完整對話，避免切掉回答或引用。
5. 使用 4-bit NF4 QLoRA 訓練。
6. 比較 SFT 前後的 held-out 回答、引用格式與拒答行為。
7. 輸出可供下一階段 DPO 繼續訓練的 LoRA adapter。

> 可使用 `PROFILE = "smoke_test"`先進行測試。流程成功後，可再依序切換成
> `colab_project` 或 `full_dataset`。

## 1. 安裝固定版本套件

若目前 runtime 已經 import 過不同版本的 Transformers／TRL，安裝完成後請重新啟動
runtime，再從下一個 cell 開始執行。

In [ ]:
%pip install -q --upgrade \
  "transformers==5.17.0" \
  "trl==1.12.0" \
  "peft==0.20.0" \
  "datasets==5.0.1" \
  "accelerate==1.14.0" \
  "bitsandbytes==0.50.2"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 45.9 MB/s eta 0:00:00


## 2. 環境檢查與共用設定

In [ ]:
import gc
import json
import random
import re
import shutil
from pathlib import Path

import bitsandbytes as bnb
import datasets
import numpy as np
import pandas as pd
import peft
import torch
import transformers
import trl
from datasets import load_dataset
from IPython.display import display
from peft import LoraConfig, PeftModel
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
)
from trl import SFTConfig, SFTTrainer

assert torch.cuda.is_available(), (
    "找不到 CUDA GPU。請在 Colab 選擇 Runtime → Change runtime type → GPU。"
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

gpu_name = torch.cuda.get_device_name(0)
gpu_major, gpu_minor = torch.cuda.get_device_capability(0)
use_bf16 = gpu_major >= 8  # A100、L4；T4 使用 fp16
compute_dtype = torch.bfloat16 if use_bf16 else torch.float16

print(f"GPU: {gpu_name}")
print(f"Compute capability: {gpu_major}.{gpu_minor}")
print(f"Training dtype: {compute_dtype}")
print(
    {
        "torch": torch.__version__,
        "transformers": transformers.__version__,
        "trl": trl.__version__,
        "peft": peft.__version__,
        "datasets": datasets.__version__,
        "bitsandbytes": bnb.__version__,
    }
)

GPU: Tesla T4
Compute capability: 7.5
Training dtype: torch.float16
{'torch': '2.11.0+cu128', 'transformers': '5.17.0', 'trl': '1.12.0', 'peft': '0.20.0', 'datasets': '5.0.1', 'bitsandbytes': '0.50.2'}


## 3. 實驗設定

| Profile | 訓練資料 | Epoch／Steps | 用途 |
|---|---:|---:|---|
| `smoke_test` | 最多 256 筆 | 30 steps | 驗證資料、模型與顯存 |
| `colab_project` | 最多 4,000 筆 | 1 epoch | Colab Pro 主要實驗 |
| `full_dataset` | 全部合格資料 | 2 epochs | 接近原 Trust-Align 設定 |

原始 Trust-Align SFT 採用 full fine-tuning 全參數微調、2 epochs、learning rate `2e-5`。本 Notebook 為配合 Colab 資源，另設最大序列長度為 2,048 tokens 並改用 4-bit QLoRA。

In [ ]:
MODEL_ID = "Qwen/Qwen3-4B-Instruct-2507"
MODEL_REVISION = "cdbee75f17c01a7cc42f958dc650907174af0554"

DATASET_ID = "declare-lab/Trust-Data"
DATASET_CONFIG = "trust_align_sft"
DATASET_REVISION = "0b89ac014bdcde23c083592af8fe6ce563b128b7"

PROFILE = "full_dataset"  # smoke_test | colab_project | full_dataset
MAX_LENGTH = 2048
RUN_DIR = "/content/qwen3-4b-trust-data-sft-run"
ADAPTER_DIR = "/content/qwen3-4b-trust-data-sft-lora"

PROFILES = {
    "smoke_test": {
        "train_limit": 256,
        "eval_limit": 64,
        "max_steps": 30,
        "num_train_epochs": 1.0,
        "eval_steps": 15,
        "generation_eval_n": 4,
    },
    "colab_project": {
        "train_limit": 4000,
        "eval_limit": 256,
        "max_steps": -1,
        "num_train_epochs": 1.0,
        "eval_steps": 100,
        "generation_eval_n": 16,
    },
    "full_dataset": {
        "train_limit": None,
        "eval_limit": 512,
        "max_steps": -1,
        "num_train_epochs": 2.0,
        "eval_steps": 200,
        "generation_eval_n": 24,
    },
}

if PROFILE not in PROFILES:
    raise ValueError(f"未知 PROFILE：{PROFILE}")

profile_cfg = PROFILES[PROFILE]
Path(RUN_DIR).mkdir(parents=True, exist_ok=True)
Path(ADAPTER_DIR).mkdir(parents=True, exist_ok=True)

print(json.dumps({"profile": PROFILE, **profile_cfg}, indent=2, ensure_ascii=False))

{
  "profile": "full_dataset",
  "train_limit": null,
  "eval_limit": 512,
  "max_steps": -1,
  "num_train_epochs": 2.0,
  "eval_steps": 200,
  "generation_eval_n": 24
}


## 4. 載入並檢查 `trust_align_sft`

此 config 的官方 schema 只有 `messages`：每筆包含一個 user prompt（指令、問題、
documents）以及一個可信任的 assistant answer。

In [ ]:
raw_dataset = load_dataset(
    path = DATASET_ID,
    name = DATASET_CONFIG,
    revision=DATASET_REVISION,
)

print(raw_dataset)
print("train columns:", raw_dataset["train"].column_names)
print("test columns:", raw_dataset["test"].column_names)

for split_name in ["train", "test"]:
    if "messages" not in raw_dataset[split_name].column_names:
        raise ValueError(f"{split_name} 缺少 messages 欄位。")


def validate_messages(example):
    messages = example["messages"]
    return (
        isinstance(messages, list)
        and len(messages) >= 2
        and messages[0].get("role") == "user"
        and messages[-1].get("role") == "assistant"
        and bool(messages[0].get("content"))
        and bool(messages[-1].get("content"))
    )


invalid_train = raw_dataset["train"].filter(
    lambda row: not validate_messages(row),
    desc="Check train message schema",
)
invalid_test = raw_dataset["test"].filter(
    lambda row: not validate_messages(row),
    desc="Check test message schema",
)

if len(invalid_train) or len(invalid_test):
    print(f"\n[診斷提示] 發現 {len(invalid_train)} 筆 train 與 {len(invalid_test)} 筆 test 不合法資料，自動執行過濾...")

    # 進行資料清理
    raw_dataset["train"] = raw_dataset["train"].filter(validate_messages, desc="Filter invalid train messages")
    raw_dataset["test"] = raw_dataset["test"].filter(validate_messages, desc="Filter invalid test messages")

    print(f"清理完成！剩餘訓練集筆數：{len(raw_dataset['train'])}，測試集筆數：{len(raw_dataset['test'])}")

sample_messages = raw_dataset["train"][0]["messages"]
print("\nUser preview:\n", sample_messages[0]["content"][:1200])
print("\nAssistant target:\n", sample_messages[-1]["content"])

DatasetDict({
    train: Dataset({
        features: ['messages'],
        num_rows: 8877
    })
    test: Dataset({
        features: ['messages'],
        num_rows: 987
    })
})
train columns: ['messages']
test columns: ['messages']

[診斷提示] 發現 4 筆 train 與 0 筆 test 不合法資料，自動執行過濾...
清理完成！剩餘訓練集筆數：8873，測試集筆數：987

User preview:
 Instruction: Write an accurate, engaging, and concise answer for the given question using only the provided search results (some of which might be irrelevant) and cite them properly. Use an unbiased and journalistic tone. Always cite for any factual claim. When citing several search results, use [1][2][3]. Cite at least one document and at most three documents in each sentence. If multiple documents support the sentence, only cite a minimum sufficient subset of the documents. If none of the provided documents contain the answer, only respond with "I apologize, but I couldn't find an answer to your question in the search results.".

Question: How are tire changes s

## 5. 使用 Qwen chat template 並過濾超長資料

先計算完整 `user + assistant` 對話長度，再排除超過 `MAX_LENGTH` 的資料。如此不會
因 truncation 靜默刪掉 target answer、citation 或前方 instruction。

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    MODEL_ID,
    revision=MODEL_REVISION,
    use_fast=True,
)

if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"


def select_candidates(dataset_split, final_limit, seed):
    shuffled = dataset_split.shuffle(seed=seed)
    if final_limit is None:
        return shuffled
    candidate_count = min(len(shuffled), max(final_limit * 2, final_limit + 128))
    return shuffled.select(range(candidate_count))


def add_token_length(example):
    token_ids = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=True,
        add_generation_prompt=False,
    )
    count = len(token_ids["input_ids"])
    return {"token_count": count}


def prepare_split(dataset_split, final_limit, seed):
    candidates = select_candidates(dataset_split, final_limit, seed)
    prepared = candidates.map(
        add_token_length,
        desc="Count complete conversation tokens",
    )
    before_filter = len(prepared)
    prepared = prepared.filter(
        lambda row: row["token_count"] <= MAX_LENGTH,
        desc=f"Keep complete conversations <= {MAX_LENGTH} tokens",
    )
    removed = before_filter - len(prepared)
    if final_limit is not None:
        prepared = prepared.select(range(min(final_limit, len(prepared))))

    print(
        f"候選 {before_filter:,} 筆；排除超長 {removed:,} 筆；"
        f"最後保留 {len(prepared):,} 筆"
    )
    return prepared

train_prepared = prepare_split(
    raw_dataset["train"], profile_cfg["train_limit"], SEED
)
eval_prepared = prepare_split(
    raw_dataset["test"], profile_cfg["eval_limit"], SEED + 1
)

if len(train_prepared) == 0 or len(eval_prepared) == 0:
    raise RuntimeError("長度過濾後沒有資料，請提高 MAX_LENGTH。")

length_summary = pd.DataFrame(
    {
        "train_tokens": pd.Series(train_prepared["token_count"]),
        "eval_tokens": pd.Series(eval_prepared["token_count"]),
    }
).describe(percentiles=[0.5, 0.9, 0.95, 0.99])
display(length_summary)

eval_view = eval_prepared
train_dataset = train_prepared.remove_columns(["token_count"])
eval_dataset = eval_prepared.remove_columns(["token_count"])

display(train_dataset[0])

Count complete conversation tokens:   0%|          | 0/8873 [00:00<?, ? examples/s]

Keep complete conversations <= 2048 tokens:   0%|          | 0/8873 [00:00<?, ? examples/s]

候選 8,873 筆；排除超長 0 筆；最後保留 8,873 筆


Count complete conversation tokens:   0%|          | 0/987 [00:00<?, ? examples/s]

Keep complete conversations <= 2048 tokens:   0%|          | 0/987 [00:00<?, ? examples/s]

候選 987 筆；排除超長 0 筆；最後保留 512 筆


,train_tokens,eval_tokens
count,8873.000000,512.000000
mean,992.016680,993.912109
std,96.391918,105.571699
min,778.000000,816.000000
50%,976.000000,976.500000
90%,1083.000000,1082.000000
95%,1139.000000,1122.600000
99%,1425.280000,1459.800000
max,1842.000000,1821.000000


{'messages': [{'content': 'Instruction: Write an accurate, engaging, and concise answer for the given question using only the provided search results (some of which might be irrelevant) and cite them properly. Use an unbiased and journalistic tone. Always cite for any factual claim. When citing several search results, use [1][2][3]. Cite at least one document and at most three documents in each sentence. If multiple documents support the sentence, only cite a minimum sufficient subset of the documents. If none of the provided documents contain the answer, only respond with "I apologize, but I couldn\'t find an answer to your question in the search results.".\n\nQuestion: Current chief justice of all provinces of pakistan?\n\nDocument [1](Title: Chief Justice of Pakistan): Chief Justice of Pakistan The Chief Justice of Pakistan (initials as CJP) is the head of the court system of Pakistan (the judicature branch of government) and the chief judge of the Supreme Court of Pakistan. The Chi

## 6. 建立 4-bit QLoRA 與 SFT 設定

`assistant_only_loss=True` 很重要：模型仍會看到完整 question 與 retrieved documents，
但 cross-entropy 只計算可信任 assistant answer，不要求模型去背誦 user prompt。
TRL 會為 Qwen3 使用能辨認 assistant token 範圍的 training chat template。

In [ ]:
quantization_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=compute_dtype,
)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

training_args = SFTConfig(
    output_dir=RUN_DIR,
    model_init_kwargs={
        "revision": MODEL_REVISION,
        "dtype": compute_dtype,
        "device_map": {"": 0},
        "attn_implementation": "sdpa",
        "low_cpu_mem_usage": True,
    },
    max_length=MAX_LENGTH,
    truncation_mode="keep_start",
    assistant_only_loss=True,
    completion_only_loss=None,
    loss_type="chunked_nll",
    packing=False,
    eval_packing=False,
    learning_rate=2e-5,
    num_train_epochs=profile_cfg["num_train_epochs"],
    max_steps=profile_cfg["max_steps"],
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=8,
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    optim="paged_adamw_8bit",
    lr_scheduler_type="cosine",
    # Transformers 5 以小於 1 的 warmup_steps 表示總步數比例。
    warmup_steps=0.1,
    max_grad_norm=1.0,
    bf16=use_bf16,
    fp16=not use_bf16,
    tf32=use_bf16,
    logging_strategy="steps",
    logging_steps=5,
    eval_strategy="steps",
    eval_steps=profile_cfg["eval_steps"],
    save_strategy="steps",
    save_steps=profile_cfg["eval_steps"],
    save_total_limit=2,
    save_only_model=True,
    report_to="none",
    dataset_num_proc=1,
    dataloader_num_workers=0,
    remove_unused_columns=True,
    seed=SEED,
    data_seed=SEED,
)

print(training_args)

SFTConfig(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
activation_offloading=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
assistant_only_loss=True,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=True,
bf16_full_eval=False,
chat_template_path=None,
completion_only_loss=None,
data_seed=42,
dataloader_drop_last=False,
dataloader_in_order=True,
dataloader_multiprocessing_context=None,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
dataset_kwargs=None,
dataset_num_proc=1,
dataset_text_field=text,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
d

## 7. 建立 SFTTrainer

這一步才會把 Qwen3 以 4-bit 載入 GPU。若 T4 發生 OOM，可先把
`MAX_LENGTH` 改為 1536，並從資料處理 cell 重新執行。

In [ ]:
gc.collect()
torch.cuda.empty_cache()

trainer = SFTTrainer(
    model=MODEL_ID,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    quantization_config=quantization_config,
    peft_config=lora_config,
)

# TRL 目前會把 QLoRA adapter 轉成 bf16；T4 沒有原生 bf16，故恢復成
# 標準 fp32 LoRA weights，再交由 fp16 autocast 執行 forward。
if not use_bf16:
    for parameter in trainer.model.parameters():
        if parameter.requires_grad:
            parameter.data = parameter.data.to(torch.float32)

trainer.model.print_trainable_parameters()
print(
    f"Allocated GPU memory: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB"
)
print(
    f"Reserved GPU memory: {torch.cuda.memory_reserved() / 1024**3:.2f} GiB"
)

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Tokenizing train dataset (num_proc=1):   0%|          | 0/8873 [00:00<?, ? examples/s]

Building labels for train dataset (num_proc=1):   0%|          | 0/8873 [00:00<?, ? examples/s]

...

Tokenizing eval dataset (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/s]

Building labels for eval dataset (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/s]

Truncating eval dataset (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset (num_proc=1):   0%|          | 0/512 [00:00<?, ? examples/s]

trainable params: 33,030,144 || all params: 4,055,498,240 || trainable%: 0.8145
Allocated GPU memory: 2.56 GiB
Reserved GPU memory: 2.66 GiB


## 8. 準備 held-out generation 並取得 SFT 前輸出

初始 LoRA 的輸出增量為零，因此這裡等價於原始 Qwen3 checkpoint。小型測試會儘量
各取一半「應回答」與「應拒答」資料。

In [ ]:
# 判定是否為應拒答的資料樣本
def target_requires_refusal(target_text):
    normalized = " ".join(target_text.lower().split())
    return (
        "couldn't find an answer" in normalized
        or "could not find an answer" in normalized
    )

# 生成回應，回應不包含輸入的 prompt_text
def generate_response(model, prompt_text, max_new_tokens=256):
    rendered = tokenizer.apply_chat_template(
        [{"role": "user", "content": prompt_text}],
        tokenize=False,
        add_generation_prompt=True,
    )
    model_inputs = tokenizer(rendered, return_tensors="pt")
    model_inputs = {key: value.to(model.device) for key, value in model_inputs.items()}

    model.eval()
    with torch.inference_mode():
        generated = model.generate(
            **model_inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            use_cache=True,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    new_tokens = generated[0, model_inputs["input_ids"].shape[1] :]
    return tokenizer.decode(new_tokens, skip_special_tokens=True).strip()

# 選取「應回答」與「應拒答」的 evaluation sample，形成 selected_indices
generation_n = min(profile_cfg["generation_eval_n"], len(eval_view))
refusal_indices = [
    index
    for index in range(len(eval_view))
    if target_requires_refusal(eval_view[index]["messages"][-1]["content"])
]
answer_indices = [
    index
    for index in range(len(eval_view))
    if not target_requires_refusal(eval_view[index]["messages"][-1]["content"])
]
target_refusal_n = generation_n // 2
selected_indices = (
    refusal_indices[:target_refusal_n]
    + answer_indices[: generation_n - target_refusal_n]
)
if len(selected_indices) < generation_n:
    used = set(selected_indices)
    selected_indices.extend(
        index for index in range(len(eval_view)) if index not in used
    )
selected_indices = selected_indices[:generation_n]

#
generation_records = []
for index in selected_indices:
    messages = eval_view[index]["messages"]
    generation_records.append(
        {
            "index": index,
            "prompt": messages[0]["content"],
            "target": messages[-1]["content"],
        }
    )

baseline_outputs = [
    generate_response(trainer.model, row["prompt"])
    for row in generation_records
]

display(
    pd.DataFrame(
        {
            "target": [row["target"] for row in generation_records],
            "before_sft": baseline_outputs,
        }
    )
)

,target,before_sft
0,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
1,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
2,"I apologize, but I couldn't find an answer to ...",Closing one eye to look at a very bright objec...
3,"I apologize, but I couldn't find an answer to ...",Hurts are an English synth-pop duo. \nCitatio...
4,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
5,"I apologize, but I couldn't find an answer to ...",It is difficult to memorize certain essential ...
6,"I apologize, but I couldn't find an answer to ...","The Pool, an online project design workspace f..."
7,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
8,"I apologize, but I couldn't find an answer to ...","Karl Valentin, Liesl Karlstadt, Erwin Faber, M..."
9,"I apologize, but I couldn't find an answer to ...","September 11, 2012, 2014, 2015, 2017 \nCitati..."


## 9. SFT 前 evaluation

這裡的 loss 是 held-out assistant tokens 的 cross-entropy。它適合比較 SFT 前後，但
不能單獨代表 citation correctness 或完整 Trust-Score。

In [ ]:
initial_eval_metrics = trainer.evaluate(metric_key_prefix="initial_eval")
initial_eval_metrics

Training Loss,Validation Loss,Step,Eval Loss,Entropy,Num Tokens,Mean Token Accuracy
No log,No log,0,2.141702,0.152309,0.000000,0.810030


{'initial_eval_loss': 2.141702175140381,
 'eval_entropy': 0.15230881497990367,
 'eval_num_tokens': 0.0,
 'eval_mean_token_accuracy': 0.8100297829369083}

## 10. 開始 SFT

此 cell 請只執行一次。訓練完成後會保存 LoRA adapter，不會複製完整 4B base model。

In [ ]:
train_result = trainer.train()
trainer.log_metrics("train", train_result.metrics)
trainer.save_metrics("train", train_result.metrics)
trainer.save_state()

# 只輸出可部署、可交接的 default adapter，不把訓練 checkpoints 一起打包。
trainer.model.save_pretrained(
    ADAPTER_DIR,
    selected_adapters=["default"],
    safe_serialization=True,
)
tokenizer.save_pretrained(ADAPTER_DIR)

training_manifest = {
    "pipeline_stage": "trust_align_sft",
    "base_model_id": MODEL_ID,
    "base_model_revision": MODEL_REVISION,
    "dataset_id": DATASET_ID,
    "dataset_config": DATASET_CONFIG,
    "dataset_revision": DATASET_REVISION,
    "profile": PROFILE,
    "max_length": MAX_LENGTH,
    "assistant_only_loss": True,
    "lora": {
        "r": lora_config.r,
        "lora_alpha": lora_config.lora_alpha,
        "lora_dropout": lora_config.lora_dropout,
        "target_modules": sorted(lora_config.target_modules),
    },
}
with open(f"{ADAPTER_DIR}/trust_align_training_manifest.json", "w") as file:
    json.dump(training_manifest, file, indent=2, ensure_ascii=False)

print(f"訓練 checkpoints：{RUN_DIR}")
print(f"可交接的 SFT LoRA adapter：{ADAPTER_DIR}")

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None, 'pad_token_id': 151643}.


Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
200,0.610984,0.468127,0.438145,1586382.000000,0.870118
400,0.601957,0.430526,0.428872,3182134.000000,0.876471
600,0.618365,0.424158,0.380409,4772635.000000,0.879517
800,0.442293,0.412906,0.410328,6359859.000000,0.878793
1000,0.619485,0.407877,0.414205,7941283.000000,0.881096
1200,0.564970,0.405276,0.380935,9514677.000000,0.880439
1400,0.513838,0.402092,0.383421,11098196.000000,0.881501
1600,0.463626,0.403758,0.384644,12689008.000000,0.881549
1800,0.377542,0.402019,0.379739,14279751.000000,0.881764
2000,0.405888,0.402098,0.381505,15867767.000000,0.882127


***** train metrics *****
  epoch                    =         2.0
  total_flos               = 360684521GF
  train_loss               =      0.6471
  train_runtime            =  4:03:31.31
  train_samples_per_second =       1.215
  train_steps_per_second   =       0.152
訓練 checkpoints：/content/qwen3-4b-trust-data-sft-run
可交接的 SFT LoRA adapter：/content/qwen3-4b-trust-data-sft-lora


## 11. SFT 後評估與前後比較

In [ ]:
final_eval_metrics = trainer.evaluate(metric_key_prefix="final_eval")


def remove_metric_prefix(metrics, prefix):
    expected = f"{prefix}_"
    return {
        (key[len(expected) :] if key.startswith(expected) else key): value
        for key, value in metrics.items()
    }


metric_comparison = pd.DataFrame(
    [
        remove_metric_prefix(initial_eval_metrics, "initial_eval"),
        remove_metric_prefix(final_eval_metrics, "final_eval"),
    ],
    index=["before_sft", "after_sft"],
).T
display(metric_comparison)

aligned_outputs = [
    generate_response(trainer.model, row["prompt"])
    for row in generation_records
]

comparison_df = pd.DataFrame(
    {
        "index": [row["index"] for row in generation_records],
        "target": [row["target"] for row in generation_records],
        "before_sft": baseline_outputs,
        "after_sft": aligned_outputs,
    }
)

comparison_path = f"{ADAPTER_DIR}/heldout_sft_generation_comparison.csv"
comparison_df.to_csv(comparison_path, index=False)
display(comparison_df)
print(f"比較結果：{comparison_path}")

Training Loss,Validation Loss,Step,Eval Loss,Entropy,Num Tokens,Mean Token Accuracy
0.594357,No log,2220,0.402656,0.381437,17604328.000000,0.882294


,before_sft,after_sft
loss,2.141702,4.026562e-01
eval_entropy,0.152309,3.814369e-01
eval_num_tokens,0.000000,1.760433e+07
eval_mean_token_accuracy,0.810030,8.822942e-01


,index,target,before_sft,after_sft
0,3,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
1,4,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
2,5,"I apologize, but I couldn't find an answer to ...",Closing one eye to look at a very bright objec...,Closing one eye allows you to look at a very b...
3,6,"I apologize, but I couldn't find an answer to ...",Hurts are an English synth-pop duo. \nCitatio...,"I apologize, but I couldn't find an answer to ..."
4,7,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
5,9,"I apologize, but I couldn't find an answer to ...",It is difficult to memorize certain essential ...,"I apologize, but I couldn't find an answer to ..."
6,10,"I apologize, but I couldn't find an answer to ...","The Pool, an online project design workspace f...","I apologize, but I couldn't find an answer to ..."
7,12,"I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ...","I apologize, but I couldn't find an answer to ..."
8,15,"I apologize, but I couldn't find an answer to ...","Karl Valentin, Liesl Karlstadt, Erwin Faber, M...","I apologize, but I couldn't find an answer to ..."
9,18,"I apologize, but I couldn't find an answer to ...","September 11, 2012, 2014, 2015, 2017 \nCitati...","I apologize, but I couldn't find an answer to ..."


比較結果：/content/qwen3-4b-trust-data-sft-lora/heldout_sft_generation_comparison.csv


## 12. 輕量引用與拒答診斷

這些是 sanity checks，不是官方 Trust-Score。它們只確認 citation 格式與 document
編號是否合理，不判斷 citation 是否真正支持相鄰 claim。

In [ ]:
def is_refusal(text):
    normalized = " ".join(text.lower().split())
    return (
        "couldn't find an answer" in normalized
        or "could not find an answer" in normalized
    )


def cited_ids(text):
    return {int(value) for value in re.findall(r"\[(\d+)\]", text)}


def available_document_ids(prompt):
    return {
        int(value)
        for value in re.findall(r"Document\s*\[(\d+)\]", prompt)
    }


def structural_diagnostics(records, outputs):
    rows = []
    for record, output in zip(records, outputs):
        target_refusal = is_refusal(record["target"])
        predicted_refusal = is_refusal(output)
        citations = cited_ids(output)
        available = available_document_ids(record["prompt"])
        rows.append(
            {
                "target_refusal": target_refusal,
                "predicted_refusal": predicted_refusal,
                "refusal_correct": target_refusal == predicted_refusal,
                "should_answer": not target_refusal,
                "has_citation": len(citations) > 0,
                "citation_ids_valid": bool(citations) and citations <= available,
            }
        )

    frame = pd.DataFrame(rows)
    answer_rows = frame[frame["should_answer"]]
    return {
        "n": len(frame),
        "refusal_accuracy": frame["refusal_correct"].mean(),
        "citation_presence_non_refusal": (
            answer_rows["has_citation"].mean() if len(answer_rows) else float("nan")
        ),
        "citation_id_validity": (
            answer_rows["citation_ids_valid"].mean()
            if len(answer_rows)
            else float("nan")
        ),
    }


diagnostic_comparison = pd.DataFrame(
    {
        "before_sft": structural_diagnostics(generation_records, baseline_outputs),
        "after_sft": structural_diagnostics(generation_records, aligned_outputs),
    }
)
display(diagnostic_comparison)

,before_sft,after_sft
n,24.000000,24.000000
refusal_accuracy,0.708333,0.833333
citation_presence_non_refusal,0.916667,0.750000
citation_id_validity,0.916667,0.750000


## 13. 封裝 SFT adapter

DPO 階段需要整個 adapter 目錄，包括：

- `adapter_config.json`
- adapter weights
- tokenizer files
- `trust_align_training_manifest.json`

In [ ]:
archive_path = shutil.make_archive(
    "/content/qwen3-4b-trust-data-sft-lora",
    "zip",
    ADAPTER_DIR,
)
print(f"SFT adapter 壓縮檔：{archive_path}")

# 在 Colab 取消下面兩行註解，可直接下載：
# from google.colab import files
# files.download(archive_path)

SFT adapter 壓縮檔：/content/qwen3-4b-trust-data-sft-lora.zip


### 選用：保存到 Google Drive

In [ ]:
SAVE_TO_DRIVE = True

if SAVE_TO_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    drive_dir = Path("/content/drive/MyDrive/llm-alignment-artifacts")
    drive_dir.mkdir(parents=True, exist_ok=True)
    destination = drive_dir / Path(archive_path).name
    shutil.copy2(archive_path, destination)
    print(f"已複製至：{destination}")
else:
    print("SAVE_TO_DRIVE=False；未寫入 Google Drive。")

Mounted at /content/drive
已複製至：/content/drive/MyDrive/llm-alignment-artifacts/qwen3-4b-trust-data-sft-lora.zip


## 15. 交接到 DPO Notebook

後續流程：

- 將 SFT 後的 LoRA Adapter 存入 Google Drive。
- 使用`trustworthy_rag_dpo_qwen3` Notebook 載入 SFT 的 LoRA Adapter 進行 DPO 訓練。

### 主要來源

- Trust-Data: https://huggingface.co/datasets/declare-lab/Trust-Data
- Trust-Align: https://github.com/declare-lab/trust-align
- TRL SFTTrainer: https://huggingface.co/docs/trl/sft_trainer
- Qwen3 model: https://huggingface.co/Qwen/Qwen3-4B-Instruct-2507